In [ ]:
!pip install -q datasets openai sentence-transformers pandas numpy tqdm scikit-learn

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
from google.colab import userdata

warnings.filterwarnings('ignore')

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()

SEED = 42

In [ ]:
dataset = load_dataset("ailsntua/QEvasion")
raw_train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

train_df, val_df = train_test_split(
    raw_train_df,
    test_size=700,
    random_state=SEED,
    stratify=raw_train_df['clarity_label']
)

val_indices = set(val_df.index.tolist())

In [ ]:
clarity_counts = train_df['clarity_label'].value_counts()
total = len(train_df)

majority_class = clarity_counts.idxmax()
majority_count = clarity_counts.max()

TARGET_RATIO = 0.6

print("Augmentation plan")
target_count = int(majority_count * TARGET_RATIO)
print(f"\nTarget: {TARGET_RATIO*100:.0f}% of majority class.")

augmentation_targets = {}
for label in ["Clear Reply", "Clear Non-Reply"]:
    current = clarity_counts.get(label, 0)
    to_generate = max(0, target_count - current)
    augmentation_targets[label] = to_generate
    print(f"\n{label}:")
    print(f"  Current in train: {current}")
    print(f"  Target: {target_count}")
    print(f"  To generate: {to_generate}")

print(f"\nTotal samples to generate: {sum(augmentation_targets.values())}")

In [ ]:
SYSTEM_PROMPT_CLEAR_REPLY = """You are an expert linguist specializing in political discourse.

Your task: Paraphrase a Question and Answer pair where the politician gives a DIRECT, CLEAR answer.

## WHAT MAKES IT A CLEAR REPLY:
- The politician DIRECTLY answers the question asked
- Provides specific information, facts, or a clear position
- No hedging, no deflection, no topic shifts
- The listener knows exactly what the answer is

## STRICT RULES:
1. Analyze and understand the rhetorical strategy used in the direct and clear reply, including aspects such as tone, wording, or overall structure.
2. You MUST paraphrase BOTH the question AND the answer
3. Keep the question length within ±20% of the original
4. Keep the answer length within ±20% of the original
5. The paraphrased answer must be EQUALLY DIRECT as the original

## OUTPUT (JSON only):
{
  "rhetorical_strategy": "Provide a short explanation of the rhetorical approach used in the direct and clear reply, including elements like tone, wording, or overall structure",
  "paraphrased_question": "Your paraphrased question",
  "paraphrased_answer": "Your paraphrased answer - must be EQUALLY DIRECT"
}"""

SYSTEM_PROMPT_CLEAR_NON_REPLY = """You are an expert linguist specializing in political discourse.

Your task: Paraphrase a Question and Answer pair where the politician EXPLICITLY REFUSES to answer.

## WHAT MAKES IT A CLEAR NON-REPLY:
- The politician clearly states they WON'T or CAN'T answer
- Explicit refusal phrases: "I can't comment", "I don't know", "I'm not going to discuss that"
- It is an OPEN refusal to answer
- The listener clearly understands that the politician is not answering the question

## STRICT RULES:
1. Analyze and understand the rhetorical strategy used in the refusal, including aspects such as tone, wording, or overall structure.
2. You MUST paraphrase BOTH the question AND the answer
3. Keep the question length within ±20% of the original
4. Keep the answer length within ±20% of the original
5. The paraphrased answer must REFUSE JUST AS CLEARLY as the original

## OUTPUT (JSON only):
{
  "rhetorical_strategy": "Provide a short explanation of the rhetorical approach used in the refusal, including elements like tone, wording, or overall structure",
  "paraphrased_question": "Your paraphrased question",
  "paraphrased_answer": "Your paraphrased answer - must REFUSE JUST AS CLEARLY"
}"""


def create_clear_reply_prompt(question: str, answer: str) -> str:
    q_len = len(question.split())
    a_len = len(answer.split())

    return f"""## PARAPHRASE THIS QUESTION AND DIRECT ANSWER PAIR

### ORIGINAL QUESTION ({q_len} words):
{question}

### ORIGINAL ANSWER ({a_len} words) - DIRECT & CLEAR:
{answer}

---

## RULES:

1. Paraphrase the question using different words but preserving the EXACT same meaning and level of directness of the initial question.
2. Identify what makes this answer clear and direct by looking for: explicit statements, specific facts, clear yes/no, definitive positions.
3. Paraphrase the answer using different words but preserving the EXACT SAME DIRECTNESS and same level of clarity and specificity of the initial answer.
"""


def create_clear_non_reply_prompt(question: str, answer: str) -> str:
    q_len = len(question.split())
    a_len = len(answer.split())

    return f"""## PARAPHRASE THIS EXPLICIT REFUSAL TO ANSWER

### ORIGINAL QUESTION ({q_len} words):
{question}

### ORIGINAL ANSWER ({a_len} words) - EXPLICIT REFUSAL:
{answer}

---

## RULES:

1. Paraphrase the question using different words but preserving the EXACT same meaning and level of directness of the initial question.
2. Identify what makes this answer a clear non-reply by looking for: explicit declinations or direct refusals.
3. Paraphrase the answer using different words but keeping the refusal just as clear and as explicit.
"""


def create_paraphrase_prompt(question: str, answer: str, clarity_label: str) -> tuple:
    if clarity_label == "Clear Reply":
        return SYSTEM_PROMPT_CLEAR_REPLY, create_clear_reply_prompt(question, answer)
    elif clarity_label == "Clear Non-Reply":
        return SYSTEM_PROMPT_CLEAR_NON_REPLY, create_clear_non_reply_prompt(question, answer)
    else:
        raise ValueError(f"Unsupported clarity label for augmentation: {clarity_label}")


def parse_gpt_response(response_text: str) -> dict:
    try:
        clean_text = response_text.strip()
        clean_text = clean_text.replace("```json", "").replace("```", "").strip()

        import re
        json_match = re.search(r'\{[^{}]*"paraphrased_question"[^{}]*"paraphrased_answer"[^{}]*\}', clean_text, re.DOTALL)
        if not json_match:
            json_match = re.search(r'\{.*\}', clean_text, re.DOTALL)

        if json_match:
            clean_text = json_match.group()

        data = json.loads(clean_text)

        if "paraphrased_question" in data and "paraphrased_answer" in data:
            return {
                "question": data["paraphrased_question"],
                "answer": data["paraphrased_answer"],
            }
        else:
            print(f"Missing keys. Found: {list(data.keys())}")
            return None

    except json.JSONDecodeError as e:
        print(f"JSON Parse Error: {e}")
        print(f"Response: {response_text[:400]}...")
        return None
    except Exception as e:
        print(f"Parse Error: {e}")
        return None

In [ ]:

MIN_SIMILARITY = 0.75
MAX_SIMILARITY = 0.95

def compute_similarity(original_text: str, augmented_text: str) -> float:
    embeddings = embedding_model.encode([original_text, augmented_text])
    similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    return similarity

def is_valid_augmentation(
    original_q: str,
    original_a: str,
    augmented_q: str,
    augmented_a: str,
    min_similarity: float = MIN_SIMILARITY,
    max_similarity: float = MAX_SIMILARITY
) -> tuple:
    q_sim = compute_similarity(original_q, augmented_q)
    a_sim = compute_similarity(original_a, augmented_a)
    original_combined = f"Question: {original_q} Answer: {original_a}"
    augmented_combined = f"Question: {augmented_q} Answer: {augmented_a}"
    combined_sim = compute_similarity(original_combined, augmented_combined)
    is_valid = min_similarity <= combined_sim <= max_similarity

    return is_valid, q_sim, a_sim, combined_sim

In [ ]:
def generate_paraphrase(
    question: str,
    answer: str,
    clarity_label: str,
    model: str = "gpt-5.1"
) -> dict:

    system_prompt, user_prompt = create_paraphrase_prompt(question, answer, clarity_label)

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
        )

        response_text = response.choices[0].message.content
        parsed = parse_gpt_response(response_text)

        return parsed

    except Exception as e:
        print(f"API error: {e}")
        return None

In [ ]:
def augment_minority_classes(
    train_df: pd.DataFrame,
    targets: dict,
    min_similarity: float = MIN_SIMILARITY,
    max_similarity: float = MAX_SIMILARITY,
    model: str = "gpt-5.1",
    max_retries: int = 5,
) -> pd.DataFrame:

    augmented_samples = []

    for label, to_generate in targets.items():

        if to_generate <= 0:
            continue

        print(f"Augmenting: {label}")
        print(f"Target: {to_generate} samples")
        print(f"Cosine similarity range: [{min_similarity}, {max_similarity}]")

        class_samples = train_df[train_df['clarity_label'] == label].copy()
        sample_idx = 0
        generated = 0

        pbar = tqdm(total=to_generate, desc=f"Generating {label}")

        while generated < to_generate:
            original = class_samples.iloc[sample_idx % len(class_samples)]
            sample_idx += 1

            for retry in range(max_retries):
                stats[label]['generated'] += 1

                result = generate_paraphrase(
                    original['question'],
                    original['interview_answer'],
                    label,
                    model=model
                )

                if result is None:
                    continue

                is_valid, q_sim, a_sim, combined_sim = is_valid_augmentation(
                    original['question'],
                    original['interview_answer'],
                    result['question'],
                    result['answer'],
                    min_similarity=min_similarity,
                    max_similarity=max_similarity
                )

                if is_valid:
                    augmented_samples.append({
                        'question': result['question'],
                        'interview_answer': result['answer'],
                        'clarity_label': label,
                        'is_augmented': True,
                        'original_idx': original.name,
                        'q_similarity': q_sim,
                        'a_similarity': a_sim,
                        'combined_similarity': combined_sim
                    })
                    generated += 1
                    pbar.update(1)
                    break

            if sample_idx > len(class_samples) * 15 and generated < to_generate:
                print(f"\nWarning: Struggled to generate valid samples for {label}")
                print(f"Generated {generated}/{to_generate}")
                break

        pbar.close()

    return pd.DataFrame(augmented_samples)

In [ ]:
augmented_df = augment_minority_classes(
    train_df=train_df,
    targets=augmentation_targets,
    min_similarity=MIN_SIMILARITY,
    max_similarity=MAX_SIMILARITY,
    model="gpt-5.1",
    max_retries=5
)

print(f"\nTotal augmented samples generated: {len(augmented_df)}")

In [ ]:
if len(augmented_df) > 0:
    original_train = train_df.copy()
    original_train['is_augmented'] = False
    original_train['original_idx'] = original_train.index
    original_train['q_similarity'] = 1.0
    original_train['a_similarity'] = 1.0
    original_train['combined_similarity'] = 1.0

    columns_to_keep = ['question', 'interview_answer', 'clarity_label', 'is_augmented',
                       'original_idx', 'q_similarity', 'a_similarity', 'combined_similarity']

    combined_train_df = pd.concat([
        original_train[columns_to_keep],
        augmented_df[columns_to_keep]
    ], ignore_index=True)


In [ ]:
import os
os.makedirs('../data', exist_ok=True)

augmented_df.to_csv('../data/augmented_samples_only.csv', index=False)
combined_train_df.to_csv('../data/train_augmented.csv', index=False)
val_df.to_csv('../data/validation_clean.csv', index=False)
augmented_df.to_json('../data/augmented_samples.json', orient='records', indent=2)

In [ ]:
original_dataset = load_dataset("ailsntua/QEvasion")
test_df = original_dataset['test'].to_pandas()

essential_columns = ['question', 'interview_answer', 'clarity_label', 'is_augmented']

train_for_hf = combined_train_df[['question', 'interview_answer', 'clarity_label', 'is_augmented']].copy()

val_for_hf = val_df[['question', 'interview_answer', 'clarity_label']].copy()
val_for_hf['is_augmented'] = False

test_for_hf = test_df[['question', 'interview_answer', 'clarity_label']].copy()
test_for_hf['is_augmented'] = False

train_dataset = Dataset.from_pandas(train_for_hf, preserve_index=False)
val_dataset = Dataset.from_pandas(val_for_hf, preserve_index=False)
test_dataset = Dataset.from_pandas(test_for_hf, preserve_index=False)

augmented_dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

augmented_dataset.save_to_disk('../data/qevasion-gpt5.1-augmented')


In [ ]:
from huggingface_hub import login

login()
augmented_dataset.push_to_hub("gabrielstefan04/qevasion-gpt5.1-augmented")